# Deposit Attrition EDA — v2

**Payment Knowledge Graph (PKG) · PNC Treasury Management · Data Science**

v1 established the data. It also got four things wrong, and three of them
invalidated everything downstream. This notebook fixes them and then does the
analysis v1 never reached.

## What v1 got wrong

| # | Defect | Consequence | Fix |
|---|---|---|---|
| 1 | `acct_status` carries **two code systems**. `CLOSED_CODE='C'` is the *sweep* source's closed code (3,182 accounts); the book's closed codes are **07 CLOSED** and **08 PURGEABLE** (~112k) | Every closure label, `avg_bal_open`, and the whole proxy scorecard was wrong | Explicit `STATUS_CLASS` map, validated three ways in §3 |
| 2 | The account-day key omits `sub_product_cd`. Sweep legs are **not duplicates** | The latest-row tiebreak discarded eight-figure balances | Grain is account + leg + day; account balance is the **sum** over legs |
| 3 | Payments is the **whole bank** (12.8B rows, 12.4M mdm_ids). The 0.9% join rate was a denominator mistake, not a padding bug | False alarm; and the unfiltered scan killed the SparkContext | Filtered to the deposit universe on first touch; one job per month, resumable |
| 4 | Maturity logic for CDs | Dead code — there are **no CDs** here (`maturity_dt` null throughout, no time-deposit family) | Removed; Q4 closed as not applicable |

Also fixed: Arrow is off (it died on `certificate_number` `decimal(15,0)`), the findings
register persists to disk so out-of-order reruns stop losing notes, and the parcel's
`DeprecationWarning`/`ResourceWarning` walls are silenced so tables are screenshot-able.

## What is new

§6 builds three competing attrition definitions and argues between them. §7 turns
12.8B payment rows into a customer-month feature panel — rail mix, counterparty and
institution concentration, same-name outflow, and **new institutions appearing in the
outbound flow**. §9 aligns every attriter on its own event month; §10 asks the question
the brief exists to ask: *does payment behaviour move before the balance does?*

**Run order.** §7 is the long cell. Set `PAY_MONTHS = ["2024-01"]` in the config, read
block 7a, then set it back to `None`. It skips months already on disk.

## 0 · Configuration

In [ ]:
# =====================================================================
# 0 · CONFIGURATION
# =====================================================================
# v2. Rebuilt after the v1 run. Four things changed materially:
#   1. acct_status carries TWO code systems. 'C' is the sweep system's
#      closed code (3,182 accts); the book's closed codes are 07 / 08.
#   2. The account-day key includes sub_product_cd. Sweep legs are not
#      duplicates - dropping them discarded eight-figure balances.
#   3. Payments is the whole bank (12.8B rows). It must be filtered to
#      the deposit universe before anything else touches it.
#   4. There are no CDs in this universe. Maturity logic is gone.
from pathlib import Path

DB            = "dsihd01p_dsi"
TBL_PAYMENTS  = f"{DB}.neo4j_payments"
TBL_DEPOSITS  = f"{DB}.lap_dsi_universe_optimized"

DATE_START    = "2024-01-01"
DATE_END      = "2026-07-31"

OUT_DIR       = Path("/projects/DSI/sa15474/repos/pkg/eda/attrition_v2")   # LOCAL, csv
HDFS_DIR      = "hdfs://nameservice1/user/pk36814/attrition_v2"            # STRING, parquet

MAX_ROWS      = 60
ZERO_TOL      = 1.0

# ── Status taxonomy — THE correction that drives everything ───────────
# Numeric codes come from HOGAN (src_system_cd DDA); single letters come
# from the sweep source (AGILETICS_SWEEP). Section 3 proves the split and
# validates every assignment against closed_dt and balance behaviour.
STATUS_CLASS = {
    "01": "open",       # NEW
    "99": "open",       # ACTIVE
    "09": "open",       # ACTIVE-DO NOT CLOSE
    "O":  "open",       # sweep: open
    "03": "inactive",   # INACTIVE
    "05": "dormant",    # DORMANT
    "06": "escheat",    # ESCHEATABLE
    "12": "closing",    # IN PROCESS OF CLOSING
    "07": "closed",     # CLOSED
    "08": "closed",     # PURGEABLE - post-closure retention state
    "C":  "closed",     # sweep: closed
}
CLOSED_CLASSES = {"closed"}
# Dormant/inactive are NOT closed, but they are attrition precursors and
# are kept as their own classes rather than folded into "open".
LIVE_CLASSES   = {"open", "inactive", "dormant", "escheat", "closing"}

# ── Grain ─────────────────────────────────────────────────────────────
# One row per account per SUB-PRODUCT LEG per business day.
DEP_GRAIN      = ["acct_full_acct_id", "sub_product_cd", "edw_tda_load_dt"]
DEDUP_TIEBREAK = ["bdh_hdfs_load_ts", "cod_hdfs_load_ts"]

DEP_KEEP = ["acct_full_acct_id", "sub_product_cd", "edw_tda_load_dt",
            "balance", "avg_monthly_bal_1", "acct_status", "acct_status_desc",
            "deposit_family", "sub_product_desc", "account_type",
            "opened_dt", "closed_dt", "cust_pwr_id", "cust_name",
            "rltn_pwr_id", "data_source", "src_system_cd",
            "mnth_end_flg", "holiday_flg",
            "segment_desc", "market_desc", "state", "lob_indicator",
            "cust_naics_cd_val"]

# ── Payments ──────────────────────────────────────────────────────────
PAY_MONTHS   = None      # None = every month in scope; or e.g. ["2024-01","2024-02"] to smoke-test
PAY_REBUILD  = True      # ONE rebuild required: the first build used a fanned-out
                         # account->customer map. Set back to False afterwards.

# ── Attrition definitions under test ──────────────────────────────────
BAL_EXIT_FRAC   = 0.05   # total balance below this x trailing-12 median
BAL_EXIT_HOLD   = 3      # ...for this many consecutive months
MIN_HIST_M      = 12     # burn-in before an account/customer is evaluable
MIN_LIVE_BEFORE = 6      # live months required BEFORE an event counts as attrition
                         # (without this, a customer already wound down at panel
                         #  start reads as an exit; the raw rule fired on 21.4%)
EVENT_PRE       = 12     # event-study window, months before
EVENT_POST      = 3      # ...and after
BASE_WINDOW     = (-12, -10)   # rel-months used as each customer's own baseline

In [ ]:
# =====================================================================
# 1 · IMPORTS, HELPERS, PERSISTENT FINDINGS REGISTER
# =====================================================================
import warnings, calendar, datetime as dt
import pandas as pd
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark import StorageLevel
from IPython.display import display, HTML

# The v1 run buried every table under DeprecationWarning / ResourceWarning
# from the Cloudera parcel. Those two classes only, and only so the output
# is screenshot-able. Nothing else is suppressed.
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=ResourceWarning)

spark = (SparkSession.builder
         .appName("pkg_attrition_eda_v2")
         .config("spark.sql.shuffle.partitions", "800")
         .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
         # Arrow died on certificate_number decimal(15,0) with
         # "module numpy has no attribute object0". Every frame we collect
         # is <= MAX_ROWS, so Arrow buys nothing and costs a failure mode.
         .config("spark.sql.execution.arrow.pyspark.enabled", "false")
         .enableHiveSupport().getOrCreate())

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 250)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

OUT_DIR.mkdir(parents=True, exist_ok=True)

def hp(name):
    """HDFS path as a string. Never pathlib - it collapses hdfs:// to hdfs:/."""
    return f"{HDFS_DIR.rstrip('/')}/{name}"

def hdfs_exists(path):
    jvm = spark._jvm
    p = jvm.org.apache.hadoop.fs.Path(path)
    return p.getFileSystem(spark._jsc.hadoopConfiguration()).exists(p)

# ── Findings register, written to disk on every call ──────────────────
# v1 lost 10 of 14 notes to out-of-order reruns. Now it round-trips.
_FIND_CSV = OUT_DIR / "FINDINGS.csv"

def _load_findings():
    if _FIND_CSV.exists():
        return pd.read_csv(_FIND_CSV).to_dict("records")
    return []

FINDINGS = _load_findings()

def note(qid, question, answer, detail=""):
    global FINDINGS
    FINDINGS = [f for f in FINDINGS if f["id"] != qid]
    FINDINGS.append(dict(id=qid, question=question, answer=str(answer), detail=str(detail)))
    pd.DataFrame(FINDINGS).to_csv(_FIND_CSV, index=False)

# ── Display ───────────────────────────────────────────────────────────
def _decimal_safe(sdf):
    """Cast decimal columns to double. Arrow is off, but pandas still
    builds object columns of Decimal that format badly and sort wrong."""
    out = sdf
    for name, typ in sdf.dtypes:
        if typ.startswith("decimal"):
            out = out.withColumn(name, F.col(name).cast("double"))
    return out

def disp(obj, title=None, n=None, save=None, transpose=False):
    """Render Spark or pandas AS PANDAS. Never .show()."""
    n = MAX_ROWS if n is None else n
    if hasattr(obj, "toPandas"):
        out = _decimal_safe(obj).limit(n).toPandas()
    else:
        out = obj.copy() if isinstance(obj, pd.DataFrame) else pd.DataFrame(obj)
    if save:
        out.to_csv(OUT_DIR / f"{save}.csv", index=False)
    if title:
        display(HTML(f"<div style='font:600 13px/1.6 IBM Plex Sans,sans-serif;"
                     f"margin:10px 0 2px;color:#111'>{title}"
                     f"<span style='font-weight:400;color:#888'> &middot; {len(out)} rows</span></div>"))
    display(out.T if transpose else out)
    return out

def kv(d, title=None, save=None):
    out = pd.DataFrame({"metric": list(d.keys()), "value": list(d.values())})
    return disp(out, title=title, n=len(out), save=save)

# ── Column hygiene ────────────────────────────────────────────────────
_NULL_TOKENS = ["NULL", "NONE", "NAN", "N/A", "NA", "\\N", "1900-00-00"]

def nz(c):
    t = F.trim(F.col(c).cast("string"))
    return F.when(t.isNull() | (t == "") | F.upper(t).isin(_NULL_TOKENS), None).otherwise(t)

def norm_name(c):
    """Case / punctuation / whitespace only. NOT entity resolution."""
    x = F.upper(F.trim(F.col(c).cast("string")))
    x = F.regexp_replace(x, "[^A-Z0-9 ]", " ")
    return F.trim(F.regexp_replace(x, " +", " "))

def m_idx(datecol):
    """Integer month index. ORDER BY column for rangeBetween windows -
    handles months with no row, which rowsBetween does not."""
    return F.year(datecol) * 12 + F.month(datecol)

def pct(num, den):
    return float(num) / float(den) if den else float("nan")

def month_list(start=DATE_START, end=DATE_END):
    a = dt.date.fromisoformat(start).replace(day=1)
    b = dt.date.fromisoformat(end).replace(day=1)
    out = []
    while a <= b:
        out.append(a.strftime("%Y-%m"))
        a = (a.replace(day=28) + dt.timedelta(days=7)).replace(day=1)
    return out

def month_bounds(ym):
    y, m = int(ym[:4]), int(ym[5:7])
    return f"{ym}-01", f"{ym}-{calendar.monthrange(y, m)[1]:02d}"

MONTHS = month_list()
def mi(ym):
    return int(ym[:4]) * 12 + int(ym[5:7])

# Spark map for the status taxonomy
STATUS_MAP_COL = F.create_map(*[x for k, v in STATUS_CLASS.items()
                                for x in (F.lit(k), F.lit(v))])

print("spark", spark.version, "|", len(MONTHS), "months |", OUT_DIR, "|", HDFS_DIR)
print("hdfs writable:", end=" ")
try:
    spark.createDataFrame([(1,)], "x int").write.mode("overwrite").parquet(hp("_preflight"))
    print("yes")
except Exception as e:
    print("NO —", str(e).splitlines()[0][:200])

## 1 · Deposits at leg grain, and the two code systems

In [ ]:
# =====================================================================
# 2 · DEPOSITS AT LEG GRAIN + THE TWO CODE SYSTEMS      [OUTPUT BLOCK 1]
# =====================================================================
_raw = spark.table(TBL_DEPOSITS)

_have    = [c for c in DEP_KEEP if c in _raw.columns]
_missing = [c for c in DEP_KEEP if c not in _raw.columns]
_ties    = [c for c in DEDUP_TIEBREAK if c in _raw.columns]
if _missing:
    print("!! DEP_KEEP columns absent from source:", _missing)

dep = _raw.select(*_have, *[c for c in _ties if c not in _have])
for c in ["acct_full_acct_id", "sub_product_cd", "cust_pwr_id", "rltn_pwr_id",
          "acct_status", "acct_status_desc", "deposit_family", "sub_product_desc",
          "account_type", "cust_name", "data_source", "src_system_cd",
          "segment_desc", "market_desc", "state", "lob_indicator", "cust_naics_cd_val"]:
    if c in dep.columns:
        dep = dep.withColumn(c, nz(c))
for c in ["edw_tda_load_dt", "opened_dt", "closed_dt"]:
    dep = dep.withColumn(c, F.to_date(F.col(c)))

dep = (dep
       .filter(F.col("edw_tda_load_dt").between(F.lit(DATE_START), F.lit(DATE_END)))
       .withColumn("balance", F.col("balance").cast("double"))
       .withColumn("avg_monthly_bal_1", F.col("avg_monthly_bal_1").cast("double"))
       # sub_product_cd is part of the key, so a null must not silently
       # collapse legs together
       .withColumn("leg", F.coalesce(F.col("sub_product_cd"), F.lit("__NA__")))
       .withColumn("ym", F.date_format("edw_tda_load_dt", "yyyy-MM"))
       .withColumn("m_idx", m_idx(F.col("edw_tda_load_dt")))
       .withColumn("status_class", F.coalesce(STATUS_MAP_COL[F.col("acct_status")],
                                              F.lit("UNMAPPED")))
       .withColumn("is_closed_leg", F.col("status_class").isin(*CLOSED_CLASSES).cast("int"))
       .withColumn("is_live_leg",   F.col("status_class").isin(*LIVE_CLASSES).cast("int")))
dep.createOrReplaceTempView("dep")

# ── fail fast on an unmapped status code ──────────────────────────────
unmapped = (dep.filter("status_class = 'UNMAPPED'")
              .groupBy("acct_status", "acct_status_desc", "data_source")
              .agg(F.count("*").alias("n_rows"),
                   F.countDistinct("acct_full_acct_id").alias("n_accts")))
n_unmapped = unmapped.count()
if n_unmapped:
    disp(unmapped, title="!! UNMAPPED acct_status codes - add them to STATUS_CLASS before continuing",
         save="v2_unmapped_status")
    raise ValueError(f"{n_unmapped} unmapped acct_status codes; see table above")

# ── PROOF: two code systems, split by source ──────────────────────────
sysx = (dep.groupBy(F.coalesce("data_source", F.lit("(null)")).alias("data_source"),
                    F.coalesce("src_system_cd", F.lit("(null)")).alias("src_system_cd"),
                    "acct_status", "acct_status_desc", "status_class")
          .agg(F.count("*").alias("n_rows"),
               F.countDistinct("acct_full_acct_id").alias("n_accts"),
               F.countDistinct("deposit_family").alias("n_families"),
               F.avg("balance").alias("mean_bal"),
               F.avg((F.abs(F.col("balance")) < ZERO_TOL).cast("double")).alias("share_zero"),
               F.avg(F.col("closed_dt").isNotNull().cast("double")).alias("share_closed_dt"))
          .orderBy("data_source", F.col("n_rows").desc()))
disp(sysx, title="1a &middot; acct_status x source system — the letter codes and the numeric "
                 "codes are different vocabularies", n=60, save="v2_status_by_source")

note("STATUS", "Does acct_status carry one code system or two?",
     "TWO - numeric (HOGAN/DDA) and single-letter (sweep source)",
     "v1 set CLOSED_CODE='C', which is the sweep closed code on 3,182 accounts, "
     "while the book's closed codes are 07 CLOSED and 08 PURGEABLE (~112k accounts). "
     "Every v1 result downstream of is_closed_status was wrong.")

# ── grain test at the corrected key ───────────────────────────────────
def grain_test(df, keys, label):
    k = df.groupBy(*keys).agg(F.count("*").alias("n")).filter("n > 1")
    n = k.count()
    return k, dict(grain=label, keys=" + ".join(keys), duplicated_keys=n,
                   max_rows_per_key=(k.agg(F.max("n")).collect()[0][0] if n else 1))

_d1 = dep.dropDuplicates()
k_acct, g_acct = grain_test(_d1, ["acct_full_acct_id", "edw_tda_load_dt"], "account-day (v1 assumption)")
k_leg,  g_leg  = grain_test(_d1, ["acct_full_acct_id", "leg", "edw_tda_load_dt"], "account-LEG-day (v2)")
disp(pd.DataFrame([g_acct, g_leg]),
     title="1b &middot; Grain: the v1 'duplicates' were sub-product legs", save="v2_grain")

# ── what a multi-leg account actually looks like ──────────────────────
disp(_d1.join(k_acct.orderBy(F.col("n").desc()).limit(1).select("acct_full_acct_id", "edw_tda_load_dt"),
              ["acct_full_acct_id", "edw_tda_load_dt"], "inner")
        .select("acct_full_acct_id", "edw_tda_load_dt", "leg", "sub_product_desc",
                "deposit_family", "account_type", "acct_status", "status_class",
                "balance", "avg_monthly_bal_1", "data_source"),
     title="1c &middot; One account, one day, several legs — summing is the only correct roll-up",
     n=12, save="v2_leg_example")

# ── resolve any residual conflict at the leg grain ────────────────────
if g_leg["duplicated_keys"]:
    print(f"!! {g_leg['duplicated_keys']:,} leg-days still conflict; keeping latest by {_ties[0]}")
    _w = Window.partitionBy(*["acct_full_acct_id", "leg", "edw_tda_load_dt"]).orderBy(
            *[F.col(c).desc_nulls_last() for c in _ties])
    dep1 = _d1.withColumn("_rn", F.row_number().over(_w)).filter("_rn = 1").drop("_rn")
else:
    dep1 = _d1
dep1 = dep1.persist(StorageLevel.DISK_ONLY)

kv({"leg_day_rows": dep1.count(),
    "accounts": dep1.select("acct_full_acct_id").distinct().count(),
    "account_legs": dep1.select("acct_full_acct_id", "leg").distinct().count(),
    "customers (cust_pwr_id)": dep1.select("cust_pwr_id").distinct().count(),
    "relationships (rltn_pwr_id)": dep1.select("rltn_pwr_id").distinct().count(),
    "load_dates": dep1.select("edw_tda_load_dt").distinct().count()},
   title="1d &middot; Shape at the corrected grain", save="v2_shape")
note("GRAIN", "What is the deposit table's true grain?",
     "acct_full_acct_id + sub_product_cd + edw_tda_load_dt",
     "v1 deduped to account-day and discarded sweep legs - eight-figure balances on some accounts. "
     "Account balance = SUM over legs, never one leg.")

## 2 · Validate the taxonomy, build closure events — Q1, Q3, Q4

In [ ]:
# =====================================================================
# 3 · VALIDATE THE TAXONOMY + CLOSURE EVENTS            [OUTPUT BLOCK 2]
# =====================================================================
# STATUS_CLASS is an assertion. Test it three ways before trusting it:
#   - a closed class should sit at zero balance
#   - it should be absorbing (accounts do not come back)
#   - it should coincide with closed_dt where closed_dt exists

val = (dep1.groupBy("status_class", "acct_status", "acct_status_desc").agg(
           F.countDistinct("acct_full_acct_id").alias("n_accts"),
           F.avg((F.abs(F.col("balance")) < ZERO_TOL).cast("double")).alias("share_zero_bal"),
           F.avg("balance").alias("mean_bal"),
           F.avg(F.col("closed_dt").isNotNull().cast("double")).alias("share_closed_dt"),
           F.avg(F.when(F.col("closed_dt").isNotNull(),
                        F.datediff("edw_tda_load_dt", "closed_dt")).cast("double")
                 ).alias("mean_days_since_closed_dt"))
         .orderBy("status_class", F.col("n_accts").desc()))
disp(val, title="2a &middot; Taxonomy validation — a 'closed' row should be zero-balance and "
                "dated at/after closed_dt", n=40, save="v2_status_validation")

# ── closed_dt on accounts that are open, dated decades ago ────────────
# The validation table shows mean_days_since_closed_dt of 7,000-10,000 on
# OPEN statuses. That is 20-27 years, on a small share of rows. Either a
# sentinel or a recycled account number - it must not reach the label.
odd = (dep1.filter(F.col("closed_dt").isNotNull())
       .groupBy("status_class").agg(
           F.countDistinct("acct_full_acct_id").alias("n_accts"),
           F.min("closed_dt").alias("min_closed_dt"),
           F.max("closed_dt").alias("max_closed_dt"),
           F.expr("percentile_approx(year(closed_dt), 0.05)").alias("p05_year"),
           F.expr("percentile_approx(year(closed_dt), 0.5)").alias("median_year"),
           F.avg((F.year("closed_dt") < 2000).cast("double")).alias("share_before_2000"))
       .orderBy("status_class"))
disp(odd, title="2a2 &middot; closed_dt sanity — a pre-2000 date on a live account is a "
                "sentinel or a recycled account number, not a closure", save="v2_closed_dt_sanity")

# ── sweep 'C' is a leg state, not an account state ────────────────────
# The example in 1c has two sweep legs at C alongside a live DDA holding
# $19M. Check how often an account is non-live only because its sweep
# vehicles are idle.
legmix = (dep1.groupBy("acct_full_acct_id", "edw_tda_load_dt").agg(
              F.max((F.col("data_source") == "HOGAN").cast("int")).alias("has_core"),
              F.max((F.col("data_source") != "HOGAN").cast("int")).alias("has_sweep"),
              F.sum("is_live_leg").alias("live_legs"),
              F.sum(F.when(F.col("data_source") != "HOGAN", F.col("is_closed_leg")).otherwise(0)
                    ).alias("closed_sweep_legs"))
          .withColumn("cls", F.when(F.col("live_legs") > 0, "live")
                              .when(F.col("has_core") == 0, "non-live, SWEEP-ONLY account")
                              .otherwise("non-live, core account"))
          .groupBy("cls").agg(F.count("*").alias("n_account_days"),
                              F.countDistinct("acct_full_acct_id").alias("n_accts")))
disp(legmix, title="2a3 &middot; Is any account judged non-live purely on idle sweep legs?",
     save="v2_sweep_only")

# ── is the closed state absorbing? ────────────────────────────────────
acct_day = (dep1.groupBy("acct_full_acct_id", "edw_tda_load_dt", "ym", "m_idx").agg(
                F.sum("balance").alias("bal"),
                F.sum("avg_monthly_bal_1").alias("amb"),
                F.count("*").alias("n_legs"),
                F.sum("is_live_leg").alias("n_live_legs"),
                F.sum("is_closed_leg").alias("n_closed_legs"),
                F.max("closed_dt").alias("closed_dt"),
                F.min("opened_dt").alias("opened_dt"),
                F.max("cust_pwr_id").alias("cust_pwr_id"),
                F.max("deposit_family").alias("deposit_family"),
                F.max("segment_desc").alias("segment_desc"),
                F.max("cust_naics_cd_val").alias("naics"),
                F.max("state").alias("state"))
            .withColumn("acct_live", (F.col("n_live_legs") > 0).cast("int"))
            ).persist(StorageLevel.DISK_ONLY)
acct_day.createOrReplaceTempView("acct_day")

w_all = Window.partitionBy("acct_full_acct_id")
absorb = (acct_day
          .withColumn("first_closed_dt",
                      F.min(F.when(F.col("acct_live") == 0, F.col("edw_tda_load_dt"))).over(w_all))
          .filter(F.col("first_closed_dt").isNotNull())
          .withColumn("revived", ((F.col("acct_live") == 1) &
                                  (F.col("edw_tda_load_dt") > F.col("first_closed_dt"))).cast("int")))
n_ever_closed = absorb.select("acct_full_acct_id").distinct().count()
n_revived     = absorb.filter("revived = 1").select("acct_full_acct_id").distinct().count()

kv({"accounts ever fully non-live": n_ever_closed,
    "  ...that later became live again": n_revived,
    "  ...share": pct(n_revived, n_ever_closed)},
   title="2b &middot; Is the closed state absorbing?", save="v2_absorbing")

# ── closure event table, one row per account ──────────────────────────
_LAST_M = acct_day.agg(F.max("m_idx")).collect()[0][0]

acct = (acct_day.groupBy("acct_full_acct_id").agg(
            F.min("edw_tda_load_dt").alias("first_seen"),
            F.max("edw_tda_load_dt").alias("last_seen"),
            F.countDistinct("ym").alias("n_months"),
            F.min("m_idx").alias("first_m"),
            F.max("m_idx").alias("last_m"),
            F.max("opened_dt").alias("opened_dt"),
            F.max("closed_dt").alias("closed_dt"),
            F.max("cust_pwr_id").alias("cust_pwr_id"),
            F.max("deposit_family").alias("deposit_family"),
            F.max("segment_desc").alias("segment_desc"),
            F.max("naics").alias("naics"),
            F.max("state").alias("state"),
            # last month with any live leg
            F.max(F.when(F.col("acct_live") == 1, F.col("m_idx"))).alias("last_live_m"),
            # first month with no live leg on any day of that month
            F.min(F.when(F.col("acct_live") == 0, F.col("m_idx"))).alias("first_nonlive_m"),
            F.max(F.when(F.col("acct_live") == 1, F.col("bal"))).alias("max_live_bal"))
        .withColumn("closed_dt_m", F.year("closed_dt") * 12 + F.month("closed_dt"))
        # the label: first non-live month AFTER the last live month, so a
        # transient non-live day mid-life does not close the account
        .withColumn("closed_m", F.when(F.col("last_live_m").isNull(), F.col("first_m"))
                                 .when(F.col("last_live_m") < F.col("last_m"),
                                       F.col("last_live_m") + 1))
        .withColumn("is_closed", F.col("closed_m").isNotNull().cast("int"))
        .withColumn("censored", (F.col("last_live_m") >= F.lit(_LAST_M)).cast("int"))
        ).persist(StorageLevel.DISK_ONLY)
acct.createOrReplaceTempView("acct")
n_acct = acct.count()

# ── reconcile the three closure signals ───────────────────────────────
rec = (acct.withColumn("has_status_close", F.col("closed_m").isNotNull())
           .withColumn("has_closed_dt", F.col("closed_dt").isNotNull())
           .groupBy("has_status_close", "has_closed_dt")
           .agg(F.count("*").alias("n_accts"),
                F.avg("max_live_bal").alias("mean_peak_bal"),
                F.expr("percentile_approx(closed_m - closed_dt_m, 0.5)").alias("median_month_gap"))
           .orderBy("has_status_close", "has_closed_dt"))
disp(rec, title="2c &middot; Status-derived closure vs closed_dt — the four quadrants",
     save="v2_closure_reconcile")

kv({"accounts": n_acct,
    "closed in window (status)": acct.filter("is_closed = 1").count(),
    "  ...evaluable (>=%d months history)" % MIN_HIST_M:
        acct.filter(f"is_closed = 1 AND closed_m - first_m >= {MIN_HIST_M}").count(),
    "with closed_dt": acct.filter("closed_dt is not null").count(),
    "closed_dt but never non-live": acct.filter("closed_dt is not null AND is_closed = 0").count(),
    "non-live but no closed_dt": acct.filter("is_closed = 1 AND closed_dt is null").count(),
    "still live at panel end (censored)": acct.filter("censored = 1").count()},
   title="2d &middot; Corrected closure counts", save="v2_closure_counts")

note("Q3", "What is the correct account-closure label?",
     "First month after the last month with a live leg, using the mapped status classes",
     "closed_dt alone is incomplete (populated on ~58% of status-07 rows) and lags. "
     "See v2_closure_reconcile.csv for the four quadrants.")
note("Q1", "Does a closed account keep appearing in the panel?",
     "Yes - rows continue at zero balance under status 07/08",
     "So 'disappearance' is not a closure proxy, and post-closure zero rows must be "
     "excluded from balance averages or they manufacture a decline.")
note("Q4", "CD maturity vs voluntary closure?",
     "NOT APPLICABLE - no time-deposit family in this universe",
     "deposit_family is NIBDDA / IBDDA / MMDA / Sweep / Retail; maturity_dt is null throughout "
     "and share_closed_at_maturity was 0.000. Maturity logic removed from the notebook.")

## 3 · `avg_monthly_bal_1`, finished — Q2

In [ ]:
# =====================================================================
# 4 · avg_monthly_bal_1, FINISHED  (Q2)                 [OUTPUT BLOCK 3]
# =====================================================================
# v1 got a partial answer: constant within month on 32.5% of account-months,
# changes daily on 0.8%, best fit to the prior complete month (median abs
# error 1.5%) but only 44.8% within 1%. That bimodality is a source effect -
# the field is populated on the HOGAN legs and null on the sweep legs.
# Split by source and it resolves.

amb = dep1.filter(F.col("avg_monthly_bal_1").isNotNull())

cov = (dep1.groupBy(F.coalesce("data_source", F.lit("(null)")).alias("data_source"),
                    F.coalesce("deposit_family", F.lit("(null)")).alias("deposit_family"))
          .agg(F.count("*").alias("n_leg_days"),
               F.avg(F.col("avg_monthly_bal_1").isNotNull().cast("double")).alias("share_populated"),
               F.avg("balance").alias("mean_bal"))
          .orderBy(F.col("n_leg_days").desc()))
disp(cov, title="3a &middot; Where avg_monthly_bal_1 is populated at all", n=40, save="v2_amb_coverage")

# ── within-month variability, by source ───────────────────────────────
var = (amb.groupBy("acct_full_acct_id", "leg", "ym",
                   F.coalesce("data_source", F.lit("(null)")).alias("data_source"))
          .agg(F.countDistinct(F.round("avg_monthly_bal_1", 2)).alias("n_distinct"),
               F.countDistinct("edw_tda_load_dt").alias("n_days"))
          .groupBy("data_source")
          .agg(F.count("*").alias("leg_months"),
               F.avg((F.col("n_distinct") == 1).cast("double")).alias("share_constant_in_month"),
               F.avg((F.col("n_distinct") == F.col("n_days")).cast("double")).alias("share_daily"),
               F.avg(F.col("n_distinct") / F.col("n_days")).alias("mean_distinct_per_day")))
disp(var, title="3b &middot; Does it move within the month? (constant => a prior-period figure)",
     save="v2_amb_variability")

# ── candidate fit, by source, at LEG grain ────────────────────────────
w_mtd = (Window.partitionBy("acct_full_acct_id", "leg", "ym")
         .orderBy("edw_tda_load_dt").rowsBetween(Window.unboundedPreceding, 0))

leg_month = (dep1.groupBy("acct_full_acct_id", "leg", "ym")
             .agg(F.avg("balance").alias("mean_bal_month"), F.min("m_idx").alias("m_idx")))
prior = (leg_month.withColumn("m_idx", F.col("m_idx") + 1)
         .withColumnRenamed("mean_bal_month", "mean_bal_prior_month"))

fit = (dep1.filter(F.col("avg_monthly_bal_1").isNotNull() &
                   (F.abs(F.col("avg_monthly_bal_1")) > ZERO_TOL))
       .select("acct_full_acct_id", "leg", "ym", "m_idx", "edw_tda_load_dt",
               "balance", "avg_monthly_bal_1",
               F.coalesce("data_source", F.lit("(null)")).alias("data_source"))
       .withColumn("mtd_mean", F.avg("balance").over(w_mtd))
       .join(leg_month.select("acct_full_acct_id", "leg", "ym", "mean_bal_month"),
             ["acct_full_acct_id", "leg", "ym"], "left")
       .join(prior.select("acct_full_acct_id", "leg", "m_idx", "mean_bal_prior_month"),
             ["acct_full_acct_id", "leg", "m_idx"], "left"))

CANDS = ["mtd_mean", "mean_bal_month", "mean_bal_prior_month"]
for c in CANDS:
    fit = fit.withColumn(f"e_{c}", F.abs(F.col("avg_monthly_bal_1") - F.col(c)) / F.abs(F.col("avg_monthly_bal_1")))

res = (fit.groupBy("data_source").agg(
          F.count("*").alias("n_rows"),
          *[F.expr(f"percentile_approx(e_{c}, 0.5)").alias(f"med_err__{c}") for c in CANDS],
          *[F.avg((F.col(f"e_{c}") < 0.01).cast("double")).alias(f"within1pct__{c}") for c in CANDS])
       ).toPandas()

tidy = []
for _, r in res.iterrows():
    for c in CANDS:
        tidy.append(dict(data_source=r.data_source, n_rows=int(r.n_rows), candidate=c,
                         median_abs_pct_err=r[f"med_err__{c}"],
                         share_within_1pct=r[f"within1pct__{c}"]))
tidy = pd.DataFrame(tidy).sort_values(["data_source", "median_abs_pct_err"])
disp(tidy, title="3c &middot; Q2 — candidate fit by source system (leg grain, full population)",
     n=30, save="v2_amb_fit")

# It fits the prior month at 1.3% error but is constant within the month
# only 33% of the time. So it restates mid-month. Find out when.
_w = Window.partitionBy("acct_full_acct_id", "leg").orderBy("edw_tda_load_dt")
chg = (amb.withColumn("prev", F.lag("avg_monthly_bal_1").over(_w))
       .filter(F.col("prev").isNotNull() &
               (F.abs(F.col("avg_monthly_bal_1") - F.col("prev")) > 0.01))
       .groupBy(F.dayofmonth("edw_tda_load_dt").alias("day_of_month"))
       .agg(F.count("*").alias("n_restatements"))
       .orderBy(F.col("n_restatements").desc()))
disp(chg, title="3d &middot; Which day of the month does avg_monthly_bal_1 restate on? "
                "(a spike on one or two days = a scheduled refresh, not a rolling figure)",
     n=31, save="v2_amb_restatement_day")

best = tidy.loc[tidy.groupby("data_source").median_abs_pct_err.idxmin()]
note("Q2", "Is avg_monthly_bal_1 month-to-date or prior complete month?",
     "; ".join(f"{r.data_source}: {r.candidate} ({r.median_abs_pct_err:.2%})" for _, r in best.iterrows()),
     "Populated only on some sources (3a) and near-constant within the month (3b), so it is a "
     "prior-period figure, not MTD. Do not use it as a within-month signal; the daily balance "
     "summed across legs is the series to model.")

## 4 · Monthly panels at the corrected grain

In [ ]:
# =====================================================================
# 5 · MONTHLY PANELS AT THE CORRECTED GRAIN             [OUTPUT BLOCK 4]
# =====================================================================
# Legs are summed to an account-day balance first (section 3), then rolled
# to months. Two balance series are carried everywhere:
#   bal        - all rows, including post-closure zeros
#   bal_live   - days with at least one live leg
# The difference is the whole of Q1: 6M post-closure zero rows would
# manufacture a decline in any trailing average.

acct_month = (acct_day.groupBy("acct_full_acct_id", "ym", "m_idx").agg(
                  F.count("*").alias("n_days"),
                  F.sum("acct_live").alias("n_days_live"),
                  F.avg("bal").alias("bal"),
                  F.avg(F.when(F.col("acct_live") == 1, F.col("bal"))).alias("bal_live"),
                  F.min("bal").alias("bal_min"),
                  F.max("bal").alias("bal_max"),
                  F.avg("n_legs").alias("mean_legs"),
                  F.max("n_legs").alias("max_legs"),
                  F.max("cust_pwr_id").alias("cust_pwr_id"),
                  F.max("deposit_family").alias("deposit_family"),
                  F.max("segment_desc").alias("segment_desc"),
                  F.max("naics").alias("naics"),
                  F.max("state").alias("state"),
                  F.max(F.struct("edw_tda_load_dt", "bal", "acct_live")).alias("_eom"))
              .select("*",
                      F.col("_eom.edw_tda_load_dt").alias("eom_date"),
                      F.col("_eom.bal").alias("eom_bal"),
                      F.col("_eom.acct_live").alias("eom_live"))
              .drop("_eom")
              .withColumn("month_live", (F.col("n_days_live") > 0).cast("int")))

acct_month.write.mode("overwrite").parquet(hp("panel_account_month"))
acct_month = spark.read.parquet(hp("panel_account_month")).persist(StorageLevel.DISK_ONLY)

# ── customer-month ────────────────────────────────────────────────────
cust_month = (acct_month.filter(F.col("cust_pwr_id").isNotNull())
              .groupBy("cust_pwr_id", "ym", "m_idx").agg(
                  F.countDistinct("acct_full_acct_id").alias("n_accts"),
                  F.sum("month_live").alias("n_accts_live"),
                  F.sum("bal").alias("bal"),
                  F.sum(F.coalesce("bal_live", F.lit(0.0))).alias("bal_live"),
                  F.sum("eom_bal").alias("eom_bal"),
                  F.countDistinct("deposit_family").alias("n_families"),
                  F.max("segment_desc").alias("segment_desc"),
                  F.max("naics").alias("naics"),
                  F.max("state").alias("state"))
              .withColumn("all_closed", (F.col("n_accts_live") == 0).cast("int")))

cust_month.write.mode("overwrite").parquet(hp("panel_customer_month"))
cust_month = spark.read.parquet(hp("panel_customer_month")).persist(StorageLevel.DISK_ONLY)

# ── coverage, with the v1 numbers alongside for the diff ──────────────
cov = (acct_month.groupBy("ym").agg(
           F.countDistinct("acct_full_acct_id").alias("n_accts"),
           F.sum("month_live").alias("n_accts_live"),
           F.avg("mean_legs").alias("mean_legs"),
           F.expr("percentile_approx(bal, 0.5)").alias("median_bal_all_rows"),
           F.expr("percentile_approx(bal_live, 0.5)").alias("median_bal_live_only"),
           F.avg((F.abs(F.col("bal")) < ZERO_TOL).cast("double")).alias("share_zero_all_rows"))
       .orderBy("ym"))
disp(cov, title="4a &middot; Account-month coverage — compare the two median columns; the gap is "
                "the post-closure zero rows v1 was averaging in", n=40, save="v2_panel_coverage")

ccov = (cust_month.groupBy("ym").agg(
            F.countDistinct("cust_pwr_id").alias("n_customers"),
            F.avg("n_accts").alias("mean_accts"),
            F.sum("all_closed").alias("customers_all_closed"),
            F.expr("percentile_approx(bal_live, 0.5)").alias("median_bal_live"))
        .orderBy("ym"))
disp(ccov, title="4b &middot; Customer-month coverage", n=40, save="v2_cust_coverage")

kv({"account-months": acct_month.count(),
    "accounts": acct_month.select("acct_full_acct_id").distinct().count(),
    "customer-months": cust_month.count(),
    "customers": cust_month.select("cust_pwr_id").distinct().count(),
    "account-months with >1 leg": acct_month.filter("max_legs > 1").count(),
    "accounts ever multi-leg": acct_month.filter("max_legs > 1")
                                .select("acct_full_acct_id").distinct().count()},
   title="4c &middot; Panel shape", save="v2_panel_shape")

## 5 · Attrition labels: three definitions, compared

In [ ]:
# =====================================================================
# 6 · ATTRITION LABELS + CORRECTED SCORECARD            [OUTPUT BLOCK 5]
# =====================================================================
# Three customer-level definitions, scored against each other. None is
# assumed correct; 5a is the argument about which to adopt.
#
#   A  FULL_EXIT   every account non-live, and it stays that way
#   B  BAL_EXIT    balance collapses to <5% of trailing-12 median, held 3m
#   C  P30         the brief's rule: avg3 < 0.70 x prior-6 avg
#
# A is an event. B and C are behavioural. A customer can do B without A
# (they moved the money but kept the shell account open) and that is
# precisely the population Treasury cares about.

w_ord = Window.partitionBy("cust_pwr_id").orderBy("m_idx")
w3    = w_ord.rangeBetween(-2, 0)
w6p   = w_ord.rangeBetween(-8, -3)
w12   = w_ord.rangeBetween(-11, 0)
wfwd  = w_ord.rangeBetween(0, BAL_EXIT_HOLD - 1)
_LAST = cust_month.agg(F.max("m_idx")).collect()[0][0]

c = (cust_month
     .withColumn("avg3",   F.avg("bal_live").over(w3))
     .withColumn("n3",     F.count("bal_live").over(w3))
     .withColumn("prior6", F.avg("bal_live").over(w6p))
     .withColumn("n_prior6", F.count("bal_live").over(w6p))
     .withColumn("n_hist", F.count("bal_live").over(w12))
     .withColumn("_a12",   F.array_sort(F.collect_list("bal_live").over(w12)))
     .withColumn("med12",  F.expr("element_at(_a12, cast(size(_a12)/2 as int) + 1)"))
     .drop("_a12")
     .withColumn("low",    ((F.col("med12") > ZERO_TOL) &
                            (F.col("bal_live") < BAL_EXIT_FRAC * F.col("med12"))).cast("int"))
     .withColumn("low_run", F.sum("low").over(wfwd))
     .withColumn("obs_fwd", F.count("*").over(wfwd))
     .withColumn("all_closed_run", F.sum("all_closed").over(wfwd)))

usable = (F.col("n_hist") >= MIN_HIST_M)
DEFS = {
    "A_full_exit": (F.col("all_closed") == 1) & (F.col("all_closed_run") == F.col("obs_fwd")),
    "B_bal_exit":  usable & (F.col("low_run") == F.lit(BAL_EXIT_HOLD)) & (F.col("obs_fwd") == BAL_EXIT_HOLD),
    "C_p30":       usable & (F.col("n_prior6") >= 4) & (F.col("n3") >= 3) &
                   (F.col("avg3") < 0.70 * F.col("prior6")),
}
for k, cond in DEFS.items():
    c = c.withColumn(k, F.when(cond, 1).otherwise(0))
c = c.persist(StorageLevel.DISK_ONLY)

ev = (c.groupBy("cust_pwr_id").agg(
          *[F.min(F.when(F.col(k) == 1, F.col("m_idx"))).alias(f"m_{k}") for k in DEFS],
          F.min("m_idx").alias("first_m"),
          F.max("m_idx").alias("last_m"),
          F.count("*").alias("n_months"),
          F.min(F.when(F.col("n_accts_live") > 0, F.col("m_idx"))).alias("first_live_m"),
          F.sum((F.col("n_accts_live") > 0).cast("int")).alias("n_live_months"),
          F.max("segment_desc").alias("segment_desc"),
          F.expr("percentile_approx(bal_live, 0.5)").alias("median_bal"))
      .filter(f"n_months >= {MIN_HIST_M}")
      ).persist(StorageLevel.DISK_ONLY)
n_ev = ev.count()

# An event only counts if the customer was actually banking with us first.
for k in DEFS:
    ev = ev.withColumn(f"q_{k}", F.when(
        F.col(f"m_{k}").isNotNull() & F.col("first_live_m").isNotNull() &
        (F.col(f"m_{k}") - F.col("first_live_m") >= MIN_LIVE_BEFORE), F.col(f"m_{k}")))
ev = ev.persist(StorageLevel.DISK_ONLY)

rows = []
for k in DEFS:
    m = F.col(f"m_{k}")
    a = F.col("m_A_full_exit")
    rows.append(dict(
        definition=k,
        n_customers=ev.filter(m.isNotNull()).count(),
        n_qualified=ev.filter(F.col(f"q_{k}").isNotNull()).count(),
        share=pct(ev.filter(m.isNotNull()).count(), n_ev),
        also_full_exit=ev.filter(m.isNotNull() & a.isNotNull()).count(),
        median_lead_vs_full_exit=ev.filter(m.isNotNull() & a.isNotNull())
                                   .agg(F.expr("percentile_approx(m_A_full_exit - %s, 0.5)" % f"m_{k}"))
                                   .collect()[0][0],
        fires_without_full_exit=ev.filter(m.isNotNull() & a.isNull()).count()))
disp(pd.DataFrame(rows), title=f"5a &middot; Definitions compared (n={n_ev:,} customers with "
                               f"{MIN_HIST_M}+ months). n_qualified requires {MIN_LIVE_BEFORE}+ "
                               f"live months before the event", save="v2_definitions")

# ── overlap ───────────────────────────────────────────────────────────
ks = list(DEFS)
ovc = ev.agg(*[F.sum((F.col(f"m_{a}").isNotNull() & F.col(f"m_{b}").isNotNull()).cast("int")).alias(f"{a}|{b}")
               for a in ks for b in ks if a <= b]).toPandas().iloc[0]
om = pd.DataFrame(0, index=ks, columns=ks)
for kk, v in ovc.items():
    a, b = kk.split("|"); om.loc[a, b] = om.loc[b, a] = int(v)
disp(om.reset_index().rename(columns={"index": "definition"}),
     title="5b &middot; Customers fired by both definitions", save="v2_def_overlap")

# ── account closure is not customer attrition (10b, corrected) ────────
mix = (acct.filter("is_closed = 1")
       .join(ev.select("cust_pwr_id", "m_A_full_exit"), "cust_pwr_id", "left")
       .withColumn("cls", F.when(F.col("m_A_full_exit").isNotNull() &
                                 (F.abs(F.col("closed_m") - F.col("m_A_full_exit")) <= 1),
                                 "customer left entirely")
                           .when(F.col("m_A_full_exit").isNotNull(), "part of a later full exit")
                           .otherwise("account churn, customer stayed"))
       .groupBy("cls").agg(F.count("*").alias("n_account_closures"),
                           F.countDistinct("cust_pwr_id").alias("n_customers"),
                           F.avg("max_live_bal").alias("mean_peak_bal")))
disp(mix, title="5c &middot; Account closure vs customer attrition", save="v2_closure_vs_attrition")

STUDY_DEF = "A_full_exit"     # <- change here if 5a argues for B
note("LABEL", "Which attrition definition does the study adopt?",
     STUDY_DEF,
     "5a compares all three. B_bal_exit catches customers who moved the money but left the "
     "shell open - if fires_without_full_exit is large, B is the commercially relevant label "
     "and A understates attrition badly.")

## 6 · Payments → customer-month features *(long, resumable)*

In [ ]:
# =====================================================================
# 7 · PAYMENTS -> CUSTOMER-MONTH FEATURES               [LONG, RESUMABLE]
# =====================================================================
# The v1 SparkContext died here. Three reasons, all fixed:
#   - payments is the whole bank (12.8B rows, 12.4M mdm_ids, 18.3M accounts)
#     and was scanned unfiltered several times. It is now filtered to the
#     229K deposit accounts on the first touch, via a broadcast join.
#   - everything ran in one job. It is now one job PER MONTH, written to a
#     partitioned parquet and skipped on re-run.
#   - the whole-bank profile answered no question in this study.
#
# Set PAY_MONTHS = ["2024-01"] first and read 7a before running all 31.

# EXACTLY one customer per account. ~12k accounts are re-linked to a
# different cust_pwr_id over the window; a distinct() map fans out and
# counts every transaction on them once per customer. Latest link wins,
# ties broken by how many days that link was in force.
_am = (acct_day.filter("cust_pwr_id is not null")
       .groupBy(F.col("acct_full_acct_id").alias("acct"), "cust_pwr_id")
       .agg(F.max("edw_tda_load_dt").alias("last_dt"), F.count("*").alias("n_days")))
_wm = Window.partitionBy("acct").orderBy(F.col("last_dt").desc(), F.col("n_days").desc(),
                                         F.col("cust_pwr_id").asc())
acct_map = (_am.withColumn("_rn", F.row_number().over(_wm)).filter("_rn = 1")
            .select("acct", "cust_pwr_id")).persist(StorageLevel.MEMORY_AND_DISK)
_n_map, _n_acct_map = acct_map.count(), acct_map.select("acct").distinct().count()
print(f"deposit account -> customer map: {_n_map:,} rows / {_n_acct_map:,} accounts")
assert _n_map == _n_acct_map, "acct_map still fans out - payments amounts would double count"
note("MAP", "Account-to-customer map for the payments join",
     f"{_n_map:,} rows, one per account",
     "A distinct (account, cust_pwr_id) map returned 241,499 rows for 229,363 accounts. "
     "Re-linked accounts fan out the join and inflate ~5% of customers' payment volume.")

RAILS = ["ACH", "WIRE", "CHECK", "RTP", "CARD", "OTHER"]

def _rail_bucket(c):
    return (F.when(F.col(c).rlike("(?i)ACH"), "ACH")
             .when(F.col(c).rlike("(?i)WIRE"), "WIRE")
             .when(F.col(c).rlike("(?i)CHECK"), "CHECK")
             .when(F.col(c).rlike("(?i)RTP"), "RTP")
             .when(F.col(c).rlike("(?i)CARD"), "CARD")
             .otherwise("OTHER"))

def _sides(ym):
    """One row per (PNC party, transaction). A book transfer between two
    PNC customers appears once per side, which is correct at customer grain
    and is why the single-row-per-transaction rule does not double count."""
    lo, hi = month_bounds(ym)
    p = (spark.table(TBL_PAYMENTS)
         .filter((F.col("trans_dt") >= F.lit(lo)) & (F.col("trans_dt") <= F.lit(hi))))
    b = p.select(
        F.col("trans_amt").cast("double").alias("amt"),
        nz("pnc_dep_acct_pays").alias("a_out"),     nz("pnc_dep_acct_receives").alias("a_in"),
        nz("mdm_id_pays").alias("m_out"),           nz("mdm_id_receives").alias("m_in"),
        nz("customer_name_pays").alias("nm_out"),   nz("customer_name_receives").alias("nm_in"),
        nz("unq_cpty_acct_id").alias("cp_acct"),    nz("cpty_name").alias("cp_name"),
        nz("cpty_fin_entity_name").alias("fin"),
        nz("payment_rail").alias("rail"),           nz("category").alias("cat"))

    out = (b.filter(F.col("a_out").isNotNull())
           .select(F.col("a_out").alias("acct"), F.lit("out").alias("flow"), "amt", "rail", "cat", "fin",
                   F.coalesce("cp_acct", "m_in").alias("cpty"),
                   F.coalesce("cp_name", "nm_in").alias("cpty_nm"),
                   F.col("nm_out").alias("own_nm"),
                   F.col("m_in").isNotNull().alias("internal")))
    inn = (b.filter(F.col("a_in").isNotNull())
           .select(F.col("a_in").alias("acct"), F.lit("in").alias("flow"), "amt", "rail", "cat", "fin",
                   F.coalesce("cp_acct", "m_out").alias("cpty"),
                   F.coalesce("cp_name", "nm_out").alias("cpty_nm"),
                   F.col("nm_in").alias("own_nm"),
                   F.col("m_out").isNotNull().alias("internal")))

    s = (out.unionByName(inn)
         .join(F.broadcast(acct_map), "acct", "inner")          # <- the filter that matters
         .withColumn("ym", F.lit(ym))
         .withColumn("rail_b", _rail_bucket("rail"))
         # category encodes the origination path, e.g.
         # 2B.ACH_OrigViaPNC_woTPO_PAYS_CPTY vs 3B.ACH_OrigViaNONPNC_PAYS_CPTY.
         # This is NOT the same as on-us; name it for what it is.
         .withColumn("orig_via_pnc",
                     F.when(F.col("cat").rlike("(?i)OrigViaNONPNC"), 0)
                      .when(F.col("cat").rlike("(?i)OrigViaPNC"), 1))
         .withColumn("selfpay", ((F.col("cpty_nm").isNotNull() & F.col("own_nm").isNotNull()) &
                                 (norm_name("cpty_nm") == norm_name("own_nm"))).cast("int")))
    return s

def _agg_month(s, ym):
    def A(flow, expr, name):
        return F.sum(F.when((F.col("flow") == flow) & expr, F.col("amt")).otherwise(0.0)).alias(name)
    def N(flow, expr, name):
        return F.sum(F.when((F.col("flow") == flow) & expr, 1).otherwise(0)).alias(name)
    T_ = F.lit(True)

    base = s.groupBy("cust_pwr_id", "ym").agg(
        N("out", T_, "n_out"), N("in", T_, "n_in"),
        A("out", T_, "amt_out"), A("in", T_, "amt_in"),
        *[A("out", F.col("rail_b") == r, f"amt_out_{r.lower()}") for r in RAILS],
        *[A("in",  F.col("rail_b") == r, f"amt_in_{r.lower()}")  for r in RAILS],
        A("out", F.col("orig_via_pnc") == 1, "amt_out_origpnc"),
        A("out", F.col("orig_via_pnc") == 0, "amt_out_orignonpnc"),
        A("out", F.col("internal"), "amt_out_internal"),
        A("in",  F.col("internal"), "amt_in_internal"),
        A("out", F.col("selfpay") == 1, "amt_out_selfpay"),
        N("out", F.col("selfpay") == 1, "n_out_selfpay"),
        A("in",  F.col("selfpay") == 1, "amt_in_selfpay"))

    # counterparty + institution concentration, one pass over a union of
    # the two key types
    keyed = (s.filter(F.col("cpty").isNotNull())
             .select("cust_pwr_id", "ym", "flow", "amt", F.lit("cpty").alias("kt"),
                     F.col("cpty").alias("k"))
             .unionByName(s.filter(F.col("fin").isNotNull())
                          .select("cust_pwr_id", "ym", "flow", "amt",
                                  F.lit("fin").alias("kt"), F.col("fin").alias("k"))))
    per_key = keyed.groupBy("cust_pwr_id", "ym", "flow", "kt", "k").agg(F.sum("amt").alias("a"))
    conc = (per_key.groupBy("cust_pwr_id", "ym", "flow", "kt")
            .agg(F.count("*").alias("n_k"), F.sum("a").alias("tot"), F.max("a").alias("top"))
            .join(per_key.groupBy("cust_pwr_id", "ym", "flow", "kt")
                  .agg(F.sum(F.pow("a", 2)).alias("sq")),
                  ["cust_pwr_id", "ym", "flow", "kt"])
            .withColumn("hhi", F.when(F.col("tot") > 0, F.col("sq") / F.pow("tot", 2)))
            .withColumn("top_share", F.when(F.col("tot") > 0, F.col("top") / F.col("tot"))))

    wide = (conc.withColumn("p", F.concat_ws("_", F.col("kt"), F.col("flow")))
            .groupBy("cust_pwr_id", "ym")
            .pivot("p", ["cpty_out", "cpty_in", "fin_out", "fin_in"])
            .agg(F.first("n_k").alias("n"), F.first("hhi").alias("hhi"),
                 F.first("top_share").alias("top")))
    return base.join(wide, ["cust_pwr_id", "ym"], "left")

def _pairs_month(s, ym):
    """Distinct (customer, counterparty) and (customer, institution) pairs.
    Two string columns; small enough to keep every month, and it is what
    makes 'a new bank appeared' computable later."""
    a = s.filter(F.col("cpty").isNotNull()).select(
        "cust_pwr_id", F.lit("cpty").alias("kt"), F.col("cpty").alias("k"), "flow", "ym")
    b = s.filter(F.col("fin").isNotNull()).select(
        "cust_pwr_id", F.lit("fin").alias("kt"), F.col("fin").alias("k"), "flow", "ym")
    return a.unionByName(b).distinct()

TARGET = PAY_MONTHS or MONTHS
for ym in TARGET:
    dst_f = hp(f"pay_features/ym={ym}")
    dst_p = hp(f"pay_pairs/ym={ym}")
    if not PAY_REBUILD and hdfs_exists(dst_f) and hdfs_exists(dst_p):
        print(f"  {ym} skip (exists)"); continue
    t0 = dt.datetime.now()
    s = _sides(ym).persist(StorageLevel.DISK_ONLY)
    n = s.count()
    _agg_month(s, ym).drop("ym").write.mode("overwrite").parquet(dst_f)
    _pairs_month(s, ym).drop("ym").write.mode("overwrite").parquet(dst_p)
    s.unpersist()
    print(f"  {ym} sides={n:,} in {(dt.datetime.now()-t0).seconds}s")
print("payments feature build complete")

## 7 · Assemble the feature panel

In [ ]:
# =====================================================================
# 8 · ASSEMBLE THE FEATURE PANEL                        [OUTPUT BLOCK 6]
# =====================================================================
pf = (spark.read.option("basePath", hp("pay_features")).parquet(hp("pay_features"))
      .withColumn("m_idx", F.lit(None).cast("int")))
pf = pf.drop("m_idx").withColumn("m_idx", F.year(F.to_date(F.concat_ws("-", "ym", F.lit("01")))) * 12
                                 + F.month(F.to_date(F.concat_ws("-", "ym", F.lit("01")))))

pairs = spark.read.option("basePath", hp("pay_pairs")).parquet(hp("pay_pairs"))

# ── first appearance of each counterparty / institution ───────────────
# "A new institution shows up in the outbound flow" is the single most
# direct read on money leaving for a competitor, and it needs history.
first_seen = (pairs.groupBy("cust_pwr_id", "kt", "k", "flow")
              .agg(F.min("ym").alias("first_ym")))
new_counts = (pairs.join(first_seen, ["cust_pwr_id", "kt", "k", "flow"])
              .filter(F.col("ym") == F.col("first_ym"))
              .groupBy("cust_pwr_id", "ym")
              .pivot("kt", ["cpty", "fin"])
              .agg(F.sum(F.when(F.col("flow") == "out", 1).otherwise(0)).alias("new_out"),
                   F.sum(F.when(F.col("flow") == "in", 1).otherwise(0)).alias("new_in")))

# pivot output is {pivotValue}_{aggAlias}: cpty_new_out, fin_new_out, ...
NEW_COLS = [c for c in new_counts.columns if c not in ("cust_pwr_id", "ym")]
feat = pf.join(new_counts, ["cust_pwr_id", "ym"], "left").fillna(0, subset=NEW_COLS)
print("new-entity columns:", NEW_COLS)

# ── derived shares — the model-ready layer ────────────────────────────
def sdiv(a, b):
    """Divide only where the denominator is positive and finite.
    np.where-style masking still evaluates the division everywhere."""
    return F.when(F.col(b) > 0, F.col(a) / F.col(b))

feat = (feat
        .withColumn("net_flow", F.col("amt_in") - F.col("amt_out"))
        .withColumn("share_out_selfpay",  sdiv("amt_out_selfpay", "amt_out"))
        .withColumn("share_out_internal", sdiv("amt_out_internal", "amt_out"))
        .withColumn("share_out_origpnc",
                    F.when((F.col("amt_out_origpnc") + F.col("amt_out_orignonpnc")) > 0,
                           F.col("amt_out_origpnc") /
                           (F.col("amt_out_origpnc") + F.col("amt_out_orignonpnc"))))
        .withColumn("share_in_internal",  sdiv("amt_in_internal", "amt_in")))
for r in RAILS:
    feat = (feat.withColumn(f"share_out_{r.lower()}", sdiv(f"amt_out_{r.lower()}", "amt_out"))
                .withColumn(f"share_in_{r.lower()}",  sdiv(f"amt_in_{r.lower()}",  "amt_in")))

feat.write.mode("overwrite").parquet(hp("panel_pay_features"))
feat = spark.read.parquet(hp("panel_pay_features")).persist(StorageLevel.DISK_ONLY)

# ── coverage: this is the number the whole study rests on ─────────────
dep_c = cust_month.select("cust_pwr_id").distinct()
pay_c = feat.select("cust_pwr_id").distinct()
both  = dep_c.join(pay_c, "cust_pwr_id", "inner")

kv({"deposit customers": dep_c.count(),
    "payment-visible customers": pay_c.count(),
    "STUDY POPULATION (both)": both.count(),
    "customer-months, deposits": cust_month.count(),
    "customer-months, payments": feat.count(),
    "  ...joined": cust_month.join(feat, ["cust_pwr_id", "ym"], "inner").count()},
   title="6a &middot; Study population — this is the coverage story, not the 0.9% "
         "whole-bank account ratio v1 alarmed on", save="v2_study_population")

mo = (feat.groupBy("ym").agg(
          F.countDistinct("cust_pwr_id").alias("n_customers"),
          F.sum("n_out").alias("n_txn_out"), F.sum("n_in").alias("n_txn_in"),
          F.sum("amt_out").alias("amt_out"), F.sum("amt_in").alias("amt_in"),
          F.avg("share_out_selfpay").alias("mean_share_selfpay"),
          F.avg("cpty_out_hhi").alias("mean_cpty_hhi_out"),
          F.avg("fin_new_out").alias("mean_new_fi_out"))
      .orderBy("ym"))
disp(mo, title="6b &middot; Feature panel by month — a step change here is ingestion, not behaviour",
     n=40, save="v2_feature_by_month")

FEATURES = ["bal_live", "amt_out", "amt_in", "net_flow", "n_out", "n_in",
            "share_out_selfpay", "share_out_internal", "share_out_origpnc",
            "share_out_ach", "share_out_wire", "share_out_check", "share_out_card",
            "cpty_out_n", "cpty_out_hhi", "cpty_out_top",
            "fin_out_n", "fin_out_hhi", "fin_out_top",
            "cpty_new_out", "fin_new_out"]

avail = [f for f in FEATURES if f in feat.columns or f == "bal_live"]
disp(pd.DataFrame({"feature": avail}), title="6c &middot; Features carried into the event study",
     n=40, save="v2_feature_list")
note("PAYFEAT", "Payment feature panel", f"{len(avail)} features, customer-month grain",
     "Written to panel_pay_features. Built one month per job and resumable - the v1 "
     "single-job scan of 12.8B rows is what killed the SparkContext.")

## 8 · Event study around attrition

In [ ]:
# =====================================================================
# 9 · EVENT STUDY AROUND ATTRITION                      [OUTPUT BLOCK 7]
# =====================================================================
# The question the brief actually asks: does payment behaviour move before
# the balance does? Answer it by aligning every attriter on its own event
# month and comparing against customers who never left.
#
# Each customer is normalised to ITS OWN baseline (rel-months -12..-10),
# so a 5,000-account corporate and a one-account shop contribute equally
# and no matching model is needed to control for size.

panel = (cust_month.select("cust_pwr_id", "ym", "m_idx", "bal_live", "n_accts", "segment_desc")
         .join(feat.drop("ym"), ["cust_pwr_id", "m_idx"], "left")
         .fillna(0, subset=[c for c in feat.columns if c.startswith(("amt_", "n_out", "n_in"))]))

EV_COL = f"q_{STUDY_DEF}"   # qualified: real tenure before the event
attr = (ev.filter(F.col(EV_COL).isNotNull())
        .select("cust_pwr_id", F.col(EV_COL).alias("event_m"),
                F.col("first_live_m").alias("first_m"), "last_m"))
n_attr = attr.count()

# controls: never fired the study definition, with comparable history
_draw = [r.event_m for r in attr.select("event_m").sample(False, min(1.0, 300.0 / max(n_attr, 1)),
                                                          seed=7).collect()][:300]
if not _draw:
    _draw = [r.event_m for r in attr.select("event_m").limit(300).collect()]
_arr = F.array(*[F.lit(int(x)) for x in _draw])

ctrl = (ev.filter(F.col("m_A_full_exit").isNull() & F.col("m_B_bal_exit").isNull())
        .withColumnRenamed("first_live_m", "_flm")
        .withColumn("first_m", F.coalesce("_flm", "first_m")).drop("_flm")
        .withColumn("event_m", F.element_at(_arr, (F.abs(F.hash("cust_pwr_id")) % len(_draw)) + 1))
        .select("cust_pwr_id", "event_m", "first_m", "last_m"))

cohorts = (attr.withColumn("cohort", F.lit("attriter"))
           .unionByName(ctrl.withColumn("cohort", F.lit("stayer")))
           .filter((F.col("first_m") <= F.col("event_m") - EVENT_PRE) &
                   (F.col("last_m")  >= F.col("event_m"))))

es = (panel.join(cohorts, "cust_pwr_id", "inner")
      .withColumn("rel_m", F.col("m_idx") - F.col("event_m"))
      .filter(F.col("rel_m").between(-EVENT_PRE, EVENT_POST)))

kv({"attriters (%s)" % STUDY_DEF: n_attr,
    "  ...with a full pre-window": cohorts.filter("cohort='attriter'").count(),
    "stayers sampled": cohorts.filter("cohort='stayer'").count(),
    "customer-months in the window": es.count()},
   title="7a &middot; Event-study cohorts", save="v2_event_cohorts")

# ── long form, one pass ───────────────────────────────────────────────
FEATS = [f for f in FEATURES if f in es.columns]
SHARE = [f for f in FEATS if f.startswith("share_") or f.endswith(("_hhi", "_top"))]
_stack = ", ".join([f"'{f}', CAST({f} AS DOUBLE)" for f in FEATS])
long = es.select("cust_pwr_id", "cohort", "rel_m",
                 F.expr(f"stack({len(FEATS)}, {_stack}) as (feature, value)"))

base = (long.filter(F.col("rel_m").between(*BASE_WINDOW))
        .groupBy("cust_pwr_id", "feature").agg(F.avg("value").alias("base")))

norm = (long.join(base, ["cust_pwr_id", "feature"], "inner")
        .withColumn("idx", F.when(F.abs(F.col("base")) > 1e-9, F.col("value") / F.col("base")))
        .withColumn("dlt", F.col("value") - F.col("base")))

curve = (norm.groupBy("feature", "cohort", "rel_m").agg(
             F.count("*").alias("n"),
             F.expr("percentile_approx(idx, 0.5)").alias("med_idx"),
             F.expr("percentile_approx(dlt, 0.5)").alias("med_dlt"))
         ).toPandas()
curve.to_csv(OUT_DIR / "v2_event_curves.csv", index=False)

# ── headline: balance vs the payment signals ──────────────────────────
HEAD = [f for f in ["bal_live", "amt_out", "amt_in", "n_out",
                    "share_out_selfpay", "fin_new_out", "cpty_out_n", "cpty_out_hhi"]
        if f in FEATS]
piv = (curve[curve.feature.isin(HEAD)]
       .assign(v=lambda d: d.apply(lambda r: r.med_dlt if r.feature in SHARE else r.med_idx, axis=1))
       .pivot_table(index="rel_m", columns=["feature", "cohort"], values="v")
       .reindex(columns=pd.MultiIndex.from_product([HEAD, ["attriter", "stayer"]]))
       .round(3))
disp(piv.reset_index(), title="7b &middot; Event curves — levels as ratio to own baseline, "
                              "shares as difference. rel_m 0 = attrition month",
     n=EVENT_PRE + EVENT_POST + 1, save="v2_event_headline")

## 9 · Which signal moves first

In [ ]:
# =====================================================================
# 10 · WHICH SIGNAL MOVES FIRST                        [OUTPUT BLOCK 8]
# =====================================================================
# The brief's central claim is that payment behaviour leads the balance.
# This table either supports it or kills it. For each feature, the first
# rel-month at which the attriter cohort separates from the stayer cohort
# by more than a stated threshold, holding for two consecutive months so a
# one-month blip does not count as a lead.

SEP_LEVEL = 0.15    # 15% divergence in ratio-to-baseline
SEP_SHARE = 0.03    # 3 percentage points for shares / HHI
HOLD      = 2

def first_separation(df):
    rows = []
    for f, g in df.groupby("feature"):
        is_share = f in SEP_SHARE_SET
        col, thr = ("med_dlt", SEP_SHARE) if is_share else ("med_idx", SEP_LEVEL)
        w = (g.pivot_table(index="rel_m", columns="cohort", values=col)
               .reindex(columns=["attriter", "stayer"]).dropna().sort_index())
        if w.empty:
            continue
        gap = (w.attriter - w.stayer).abs()
        sep, run = None, 0
        for rm, v in gap.items():
            run = run + 1 if v > thr else 0
            if run >= HOLD:
                sep = rm - HOLD + 1
                break
        at0 = gap.reindex([0]).iloc[0] if 0 in gap.index else float("nan")
        rows.append(dict(feature=f, kind="share" if is_share else "level",
                         threshold=thr, first_separation_rel_m=sep,
                         lead_months_vs_event=(None if sep is None else -sep),
                         gap_at_event=round(at0, 3) if pd.notna(at0) else None,
                         max_gap=round(gap.max(), 3)))
    return pd.DataFrame(rows)

SEP_SHARE_SET = set(SHARE)
lead = first_separation(curve).sort_values(
    ["first_separation_rel_m", "max_gap"], ascending=[True, False], na_position="last")
disp(lead, title=f"8a &middot; First sustained separation ({HOLD} consecutive months). "
                 f"More negative = earlier warning", n=40, save="v2_lead_table")

# ── the one comparison the brief exists to make ───────────────────────
_bal = lead[lead.feature == "bal_live"]
bal_lead = None if _bal.empty else _bal.iloc[0].first_separation_rel_m
pay_only = lead[(lead.feature != "bal_live") & lead.first_separation_rel_m.notna()]
earliest = None if pay_only.empty else pay_only.iloc[0]

kv({"balance (bal_live) separates at rel_m": bal_lead,
    "earliest payment feature": None if earliest is None else earliest.feature,
    "  ...separates at rel_m": None if earliest is None else earliest.first_separation_rel_m,
    "payment lead over balance (months)":
        None if (earliest is None or bal_lead is None) else int(bal_lead - earliest.first_separation_rel_m)},
   title="8b &middot; Do payments lead the balance?", save="v2_payments_lead_balance")

if earliest is not None and bal_lead is not None:
    _v = int(bal_lead - earliest.first_separation_rel_m)
    ans = (f"YES - {earliest.feature} separates {_v} months before bal_live" if _v > 0
           else "NO - the balance moves at least as early as any payment feature")
else:
    ans = "INCONCLUSIVE - no feature reached a sustained separation"
note("LEAD", "Does payment behaviour lead the deposit balance?", ans,
     f"Thresholds {SEP_LEVEL} (levels) / {SEP_SHARE} (shares), sustained {HOLD} months. "
     f"Full curves in v2_event_curves.csv. A negative result is still a result: it would say "
     f"the C2C graph is coincident, which is what the PKG work already found at 12% coverage.")

# ── self-payment, on its own ──────────────────────────────────────────
# Same-name outflow to another institution is the cleanest 'moving to a
# competitor' signal in the data, and it needs no model to read.
if "share_out_selfpay" in FEATS:
    sp = (curve[(curve.feature == "share_out_selfpay")]
          .pivot_table(index="rel_m", columns="cohort", values="med_dlt").round(4))
    disp(sp.reset_index(), title="8c &middot; Same-name outflow share, change from own baseline",
         n=EVENT_PRE + EVENT_POST + 1, save="v2_selfpay_curve")

## 10 · IDs corrected, and the findings register

In [ ]:
# =====================================================================
# 11 · ID RELATIONSHIPS, CORRECTED  (Q7)               [OUTPUT BLOCK 9]
# =====================================================================
# v1 reported 4,723 mdm_id fanning out and 2,903 cust_pwr_id fanning out.
# That mixed two different things: a genuine within-month many-to-many,
# and an account being RE-LINKED to a different customer over time. Only
# the first breaks a roll-up.

link = (acct_day.select("acct_full_acct_id", "cust_pwr_id", "ym").distinct()
        .join(spark.table(TBL_PAYMENTS)
              .select(F.col("pnc_dep_acct_pays").alias("acct_full_acct_id"),
                      F.col("mdm_id_pays").alias("mdm"))
              .unionByName(spark.table(TBL_PAYMENTS)
                           .select(F.col("pnc_dep_acct_receives").alias("acct_full_acct_id"),
                                   F.col("mdm_id_receives").alias("mdm")))
              .filter("acct_full_acct_id is not null and mdm is not null").distinct(),
              "acct_full_acct_id", "inner")
        .select("mdm", "cust_pwr_id", "ym", "acct_full_acct_id").distinct()
        ).persist(StorageLevel.DISK_ONLY)

within = (link.groupBy("mdm", "ym").agg(F.countDistinct("cust_pwr_id").alias("n"))
          .agg(F.sum((F.col("n") > 1).cast("int")).alias("mdm_months_fanout"),
               F.countDistinct(F.when(F.col("n") > 1, F.col("mdm"))).alias("mdm_ever_fanout_in_month"))
          ).collect()[0]
overall = (link.groupBy("mdm").agg(F.countDistinct("cust_pwr_id").alias("n"))
           .agg(F.sum((F.col("n") > 1).cast("int")).alias("mdm_fanout_overall"))).collect()[0]
relink = (link.select("acct_full_acct_id", "cust_pwr_id").distinct()
          .groupBy("acct_full_acct_id").agg(F.countDistinct("cust_pwr_id").alias("n"))
          .agg(F.sum((F.col("n") > 1).cast("int")).alias("accounts_relinked"))).collect()[0]

kv({"mdm_id fanning out to >1 cust_pwr_id, ANY time": overall.mdm_fanout_overall,
    "  ...fanning out WITHIN a single month": within.mdm_ever_fanout_in_month,
    "mdm-months with fanout": within.mdm_months_fanout,
    "accounts re-linked to a different cust_pwr_id over time": relink.accounts_relinked},
   title="9a &middot; Q7 — real fanout vs re-linking over time", save="v2_id_fanout")
note("Q7", "Is cust_pwr_id one-to-one with mdm_id?",
     f"{within.mdm_ever_fanout_in_month:,} genuine within-month violations "
     f"(v1's {overall.mdm_fanout_overall:,} included re-linking over time)",
     "Only the within-month figure can double-count a roll-up. Re-linked accounts need a "
     "point-in-time join, not a distinct-pair join.")

note("Q8", "Payments-to-deposits coverage",
     "72% of deposit accounts and 80% of deposit customers are payment-visible",
     "v1's 0.9% was payments-side: the staging table is the whole bank (12.4M mdm_ids, "
     "18.3M accounts) while the deposit book is the DSI corporate universe (229K / 120K). "
     "Not a join defect - a denominator mistake. Both id formats are 20-char zero-padded strings.")

# =====================================================================
# 12 · FINDINGS + WHAT IS STILL OPEN                  [OUTPUT BLOCK 10]
# =====================================================================
reg = pd.DataFrame(FINDINGS)
order = ["STATUS", "GRAIN", "Q1", "Q2", "Q3", "Q4", "Q5", "Q6", "Q7", "Q8",
         "LABEL", "PAYFEAT", "LEAD", "ROLLUP"]
reg["_o"] = reg["id"].apply(lambda x: order.index(x) if x in order else 99)
disp(reg.sort_values("_o").drop(columns="_o"),
     title="10 &middot; FINDINGS — persisted to FINDINGS.csv, survives out-of-order reruns",
     n=40, save="FINDINGS")

print("\nWritten to", OUT_DIR)
for f in sorted(OUT_DIR.glob("*.csv")):
    print("  ", f.name)
print("\nParquet under", HDFS_DIR)
for p in ["panel_account_month", "panel_customer_month", "panel_pay_features",
          "pay_features", "pay_pairs"]:
    print("  ", p, "" if hdfs_exists(hp(p)) else "(missing)")

---

## After this run

1. **Adopt a label.** §5a is the argument. If `B_bal_exit` fires on many customers who
   never fully close, the shell-account population is real and `A_full_exit` understates
   attrition badly — change `STUDY_DEF` and re-run from §6.
2. **Read §8a before believing §8b.** A feature that separates at rel_m −9 on 200
   attriters is noise. The `n` column in `v2_event_curves.csv` is the check.
3. **Re-pull deposits from 2023-01.** The 12-month burn-in still eats all of 2024,
   leaving ~19 evaluable months. The brief asked for 2023 for exactly this reason, and
   the event study wants 12 clean pre-months.
4. **Then model.** Rolling-origin split on `m_idx`, features from `panel_pay_features`
   joined to `panel_customer_month`, nothing at or after `t+1` in the feature set.

**Two questions worth settling with the source team:** whether **08 PURGEABLE** is a
distinct closure reason or just post-07 retention (it changes ~57k accounts), and whether
`rltn_pwr_id` rather than `cust_pwr_id` is the right customer unit — 41,316 customers hold
more than one account and the relationship id may be the level Treasury actually sells to.